# Fine-Tune GPT OSS 20B with Unsloth

This notebook demonstrates how to fine-tune GPT OSS 20B for human-like responses using Unsloth.

## Setup

First, make sure you have installed all dependencies:
```bash
pip install -r requirements.txt
```

## 1. Data Preparation

In [ ]:
import json
import yaml
from pathlib import Path

# Load and inspect example data
with open('data/raw/example_data.jsonl', 'r') as f:
    examples = [json.loads(line) for line in f]

print(f"Number of examples: {len(examples)}")
print("\nFirst example:")
print(json.dumps(examples[0], indent=2))

In [ ]:
# Prepare data for training
!python scripts/prepare_data.py \
    --input data/raw/example_data.jsonl \
    --output-dir data/processed \
    --eval-ratio 0.2

## 2. Load Configuration

In [ ]:
# Load training configuration
with open('config/training_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Training Configuration:")
print(f"Model: {config['model']['name']}")
print(f"Max Sequence Length: {config['model']['max_seq_length']}")
print(f"LoRA Rank: {config['lora']['r']}")
print(f"Learning Rate: {config['training']['learning_rate']}")
print(f"Batch Size: {config['training']['per_device_train_batch_size']}")
print(f"Epochs: {config['training']['num_train_epochs']}")

## 3. Initialize Model with Unsloth

**Note:** This requires significant GPU memory. Ensure you have a GPU with at least 24GB VRAM.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Initialize model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config['model']['name'],
    max_seq_length=config['model']['max_seq_length'],
    dtype=config['model']['dtype'],
    load_in_4bit=config['model']['load_in_4bit'],
)

print("Model loaded successfully!")
print(f"Device: {next(model.parameters()).device}")

## 4. Add LoRA Adapters

In [ ]:
# Add LoRA adapters for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=config['lora']['r'],
    target_modules=config['lora']['target_modules'],
    lora_alpha=config['lora']['lora_alpha'],
    lora_dropout=config['lora']['lora_dropout'],
    bias=config['lora']['bias'],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters added successfully!")

## 5. Load and Format Dataset

In [ ]:
from datasets import load_dataset

# Load dataset
dataset = load_dataset('json', data_files={
    'train': config['dataset']['train_file'],
    'eval': config['dataset']['eval_file']
})

print(f"Training examples: {len(dataset['train'])}")
print(f"Evaluation examples: {len(dataset['eval'])}")

In [ ]:
# Format dataset with prompt template
def format_prompts(examples):
    texts = []
    for instruction, response in zip(examples['instruction'], examples['response']):
        text = config['dataset']['prompt_template'].format(
            instruction=instruction,
            response=response
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("Dataset formatted successfully!")

## 6. Start Training

You can either train in this notebook or use the training script:
```bash
python scripts/finetune.py --config config/training_config.yaml
```

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Setup training arguments
training_args = TrainingArguments(
    output_dir=config['training']['output_dir'],
    num_train_epochs=config['training']['num_train_epochs'],
    per_device_train_batch_size=config['training']['per_device_train_batch_size'],
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
    warmup_steps=config['training']['warmup_steps'],
    learning_rate=config['training']['learning_rate'],
    fp16=config['training']['fp16'],
    bf16=config['training']['bf16'],
    logging_steps=config['training']['logging_steps'],
    save_steps=config['training']['save_steps'],
    save_total_limit=config['training']['save_total_limit'],
    optim=config['training']['optim'],
    weight_decay=config['training']['weight_decay'],
    max_grad_norm=config['training']['max_grad_norm'],
    lr_scheduler_type=config['training']['lr_scheduler_type'],
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['eval'],
    dataset_text_field="text",
    max_seq_length=config['model']['max_seq_length'],
    args=training_args,
)

print("Trainer initialized. Starting training...")
# Uncomment to start training
# trainer.train()

## 7. Save Fine-tuned Model

In [ ]:
# Save the model (after training)
output_path = "./models/fine_tuned_model"
Path(output_path).mkdir(parents=True, exist_ok=True)

# Uncomment after training
# model.save_pretrained(output_path)
# tokenizer.save_pretrained(output_path)
# print(f"Model saved to {output_path}")

## 8. Inference with Fine-tuned Model

In [ ]:
# Load fine-tuned model for inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./models/fine_tuned_model",
    max_seq_length=config['model']['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

# Set to inference mode
FastLanguageModel.for_inference(model)
print("Model ready for inference!")

In [ ]:
# Generate response
def generate_response(instruction):
    prompt = f"""### Instruction:
{instruction}

### Response:
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=config['generation']['max_new_tokens'],
            temperature=config['generation']['temperature'],
            top_p=config['generation']['top_p'],
            top_k=config['generation']['top_k'],
            repetition_penalty=config['generation']['repetition_penalty'],
            do_sample=True,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

# Test the model
test_instruction = "What makes a good cup of coffee?"
print(f"Instruction: {test_instruction}\n")
print(f"Response: {generate_response(test_instruction)}")

## Next Steps

1. Collect more training data for better performance
2. Experiment with different hyperparameters
3. Adjust the prompt template for your use case
4. Evaluate model performance on your validation set
5. Share your fine-tuned model with the community!